In [901]:
import glob
import pandas as pd
%pip install Unidecode

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


# Load data

In [902]:
def load_evidence(fn):
    ds_name = fn.split("/")[-3]
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    extracted_df = df['extracted'].apply(pd.Series).add_prefix('extracted_')
    # derived_df = df['derived'].apply(pd.Series).add_prefix('derived_')
    # Combine the original DataFrame with the unwrapped columns
    # df = pd.concat([df.drop(columns=['source', 'extracted', 'derived']), source_df, extracted_df, derived_df], axis=1)
    df = pd.concat([df.drop(columns=['source', 'extracted']), source_df, extracted_df], axis=1)
    df["ds"] = ds_name
    del df["derived"]
    return df

In [903]:
fns = glob.glob("../../data/*/evidence_human/evidence.json")


In [904]:
dfs = []
for fn in fns:
    dfs.append(load_evidence(fn))
df = pd.concat(dfs)

# Perform token alignment

In [905]:
tdf = df.query("source_type == 'text'").copy()

In [906]:
tdf.head(2)

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type,ds
0,text,Monocyte 1 strongly expressed FCGR3A (CD16) an...,text,homo_sapiens,Monocyte 1,kidney,none,FCGR3A,gene,kidney_Wu2018
1,text,"Interestingly, ABCA1, which mediates sterol ef...",text,homo_sapiens,monocyte 1,kidney,none,ABCA1,gene,kidney_Wu2018


In [907]:
from extract.extract import align_ng, norm_text, reconstruct_target_by_token

In [908]:
# find the cell type
tdf["found_aln_cell_type_label"] = tdf.apply(
    lambda x: align_ng(x["source_rationale"], x["extracted_cell_type_label"])[0], axis=1
)
# reconstruct the cell type
tdf["source_cell_type_label"] = tdf["found_aln_cell_type_label"].apply(lambda x: reconstruct_target_by_token("", x[0] if len(x)>0 else ""))

# find the feature name
tdf["found_aln_feature_name"] = tdf.apply(
    lambda x: align_ng(x["source_rationale"], x["extracted_feature_name"])[0], axis=1
)
# reconstruct the feature name
tdf["source_feature_name"] = tdf["found_aln_feature_name"].apply(lambda x: reconstruct_target_by_token("", x[0] if len(x)>0 else ""))



# tdf["source_cell_type_label"] = tdf.apply(
#     lambda x: align_ng(norm_text(x["source_rationale"]), norm_text(x["extracted_cell_type_label"]))[0], 
#     axis=1)
# tdf["source_feature_name"]    = tdf.apply(
#     lambda x: align_ng(norm_text(x["source_rationale"]), norm_text(x["extracted_feature_name"]))[0], 
#     axis=1)

In [909]:
tdf["found_feature_name"] = tdf["source_feature_name"].apply(norm_text) == tdf["extracted_feature_name"].apply(norm_text)
tdf["found_cell_type_label"] = tdf["source_cell_type_label"].apply(norm_text) == tdf["extracted_cell_type_label"].apply(norm_text)

In [910]:

temp = tdf[tdf["source_cell_type_label"] != tdf["extracted_cell_type_label"]]
temp = temp[temp["ds"] == "pancreas_Segerstolpe2016"]
temp[temp["source_type"] == "text"]

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type,ds,found_aln_cell_type_label,source_cell_type_label,found_aln_feature_name,source_feature_name,found_feature_name,found_cell_type_label
18,text,"For example, in α cells we detected known mark...",text,homo_sapiens,α cells,pancreas,none,GCG,gene,pancreas_Segerstolpe2016,"[[{'token': ' a', 'enc_token': 264, 'start_idx...",a cells,"[[{'token': ' G', 'enc_token': 480, 'start_idx...",GCG,True,True
19,text,"For example, in α cells we detected known mark...",text,homo_sapiens,α cells,pancreas,none,LOXL4,gene,pancreas_Segerstolpe2016,"[[{'token': ' a', 'enc_token': 264, 'start_idx...",a cells,"[[{'token': ' LO', 'enc_token': 5125, 'start_i...",LOXL4,True,True
20,text,"For example, in α cells we detected known mark...",text,homo_sapiens,α cells,pancreas,none,DPP4,gene,pancreas_Segerstolpe2016,"[[{'token': ' a', 'enc_token': 264, 'start_idx...",a cells,"[[{'token': ' D', 'enc_token': 423, 'start_idx...",DPP4,True,True
21,text,"For example, in α cells we detected known mark...",text,homo_sapiens,α cells,pancreas,none,GC,gene,pancreas_Segerstolpe2016,"[[{'token': ' a', 'enc_token': 264, 'start_idx...",a cells,"[[{'token': ' GC', 'enc_token': 23186, 'start_...",GC,True,True
22,text,"For example, in α cells we detected known mark...",text,homo_sapiens,α cells,pancreas,none,FAP,gene,pancreas_Segerstolpe2016,"[[{'token': ' a', 'enc_token': 264, 'start_idx...",a cells,"[[{'token': ' F', 'enc_token': 435, 'start_idx...",FAP,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,text,Our data also revealed several important genes...,text,homo_sapiens,β cells,pancreas,type 2 diabetes,LEPROTL1,gene,pancreas_Segerstolpe2016,"[[{'token': ' b', 'enc_token': 293, 'start_idx...",b cells,"[[{'token': ' LE', 'enc_token': 11396, 'start_...",LEPROTL1,True,True
120,text,The expression of RGS4 and CHRM3 in pancreas h...,text,homo_sapiens,α,pancreas,type 2 diabetes,RGS4,gene,pancreas_Segerstolpe2016,"[[{'token': ' a', 'enc_token': 264, 'start_idx...",a,"[[{'token': ' R', 'enc_token': 432, 'start_idx...",RGS4,True,True
121,text,he expression of RGS4 and CHRM3 in pancreas ha...,text,homo_sapiens,α,pancreas,type 2 diabetes,CHRM3,gene,pancreas_Segerstolpe2016,"[[{'token': ' a', 'enc_token': 264, 'start_idx...",a,"[[{'token': ' CH', 'enc_token': 6969, 'start_i...",CHRM3,True,True
122,text,The expression of RGS4 and CHRM3 in pancreas h...,text,homo_sapiens,γ cells,pancreas,type 2 diabetes,RGS4,gene,pancreas_Segerstolpe2016,"[[{'token': ' g', 'enc_token': 342, 'start_idx...",g cells,"[[{'token': ' R', 'enc_token': 432, 'start_idx...",RGS4,True,True


In [911]:
agg = tdf.groupby("ds").agg({
    "found_feature_name": ["count", "sum"], 
    "found_cell_type_label": ["count", "sum"], 
    }).rename(columns={"count": "total", "sum": "found"}, level=1)
agg[("found_feature_name", "pct")] = 100*agg[("found_feature_name", "found")] / agg[("found_feature_name", "total")]
agg[("found_cell_type_label", "pct")] = 100*agg[("found_cell_type_label", "found")] / agg[("found_cell_type_label", "total")]
agg[("found_feature_name", "complete")] = agg[("found_feature_name", "pct")] == 100
agg[("found_cell_type_label", "complete")] = agg[("found_cell_type_label", "pct")] == 100
agg["all_complete"] = agg[("found_feature_name", "complete")] & agg[("found_cell_type_label", "complete")]

In [912]:
# Add a total row at the bottom of the DataFrame
total_row = agg.sum(numeric_only=True)
total_row[("found_feature_name", "pct")] = 100 * total_row[("found_feature_name", "found")] / total_row[("found_feature_name", "total")]
total_row[("found_cell_type_label", "pct")] = 100 * total_row[("found_cell_type_label", "found")] / total_row[("found_cell_type_label", "total")]

# Handling boolean columns for completeness
total_row[("found_feature_name", "complete")] = total_row[("found_feature_name", "pct")] == 100
total_row[("found_cell_type_label", "complete")] = total_row[("found_cell_type_label", "pct")] == 100
total_row["all_complete"] = total_row[("found_feature_name", "complete")] & total_row[("found_cell_type_label", "complete")]

# Add the total row to the DataFrame
agg.loc["total"] = total_row

# Sort the columns
agg = agg.sort_index(axis=1)

agg.reset_index()

/var/folders/z0/_8t5n4nd09g4xqzvkth48kp00000gn/T/ipykernel_43203/1931357756.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  total_row[("found_feature_name", "complete")] = total_row[("found_feature_name", "pct")] == 100


ds all_complete found_cell_type_label          \
                                                       complete   found   
0          adipose_Emont2022         True                  True    17.0   
1       adipose_Hildreth2021         True                  True    72.0   
2         adipose_Jaitin2019         True                  True    49.0   
3        adipose_Massier2023         True                  True    29.0   
4        adipose_Merrick2019         True                  True    29.0   
5          adipose_Vijay2019         True                  True    79.0   
6             bladder_Yu2019         True                  True    41.0   
7                bone_He2021         True                  True    72.0   
8      cortex_Booeshaghi2021         True                  True     4.0   
9             eye_Gautam2021         True                  True    25.0   
10          heart_Tucker2020         True                  True    73.0   
11             kidney_Wu2018         True                  True    31.0   
12        liver_Guillams2022         True                  True     5.0   
13            lung_Adams2020         True                  True    37.0   
14          ovary_Wagner2020         True                  True    65.0   
15  pancreas_Segerstolpe2016         True                  True   124.0   
16          placenta_Liu2018         True                  True   102.0   
17         testis_Shamis2020         True                  True   116.0   
18           yolksac_Goh2023         True                  True    96.0   
19                     total         True                  True  1066.0   

                  found_feature_name                         
      pct   total           complete   found    pct   total  
0   100.0    17.0               True    17.0  100.0    17.0  
1   100.0    72.0               True    72.0  100.0    72.0  
2   100.0    49.0               True    49.0  100.0    49.0  
3   100.0    29.0               True    29.0  100.0    29.0  
4   100.0    29.0               True    29.0  100.0    29.0  
5   100.0    79.0               True    79.0  100.0    79.0  
6   100.0    41.0               True    41.0  100.0    41.0  
7   100.0    72.0               True    72.0  100.0    72.0  
8   100.0     4.0               True     4.0  100.0     4.0  
9   100.0    25.0               True    25.0  100.0    25.0  
10  100.0    73.0               True    73.0  100.0    73.0  
11  100.0    31.0               True    31.0  100.0    31.0  
12  100.0     5.0               True     5.0  100.0     5.0  
13  100.0    37.0               True    37.0  100.0    37.0  
14  100.0    65.0               True    65.0  100.0    65.0  
15  100.0   124.0               True   124.0  100.0   124.0  
16  100.0   102.0               True   102.0  100.0   102.0  
17  100.0   116.0               True   116.0  100.0   116.0  
18  100.0    96.0               True    96.0  100.0    96.0  
19  100.0  1066.0               True  1066.0  100.0  1066.0

In [913]:
nobs = tdf.shape[0]
found_ct = tdf["found_cell_type_label"].sum()
found_fn = tdf["found_feature_name"].sum()
frac_ct = found_ct / nobs * 100
frac_fn = found_fn / nobs * 100


In [914]:
# find specific datasets with missing cell type or feature name
ds = 'cortex_Booeshaghi2021'
tdf.query(f"ds == '{ds}' & (~found_cell_type_label | ~found_feature_name)")[["source_rationale", "extracted_cell_type_label", "source_cell_type_label", "extracted_feature_name", "source_feature_name"]]

,source_rationale,extracted_cell_type_label,source_cell_type_label,extracted_feature_name,source_feature_name
